# Fixation mRNN Loss Sweep

This notebook tests whether additional loss terms help the condition-only Elman mRNN recover higher-frequency temporal variations in the 42-PC neural trajectories.

Fixed modeling decisions:

- Input is only the 3 fixation-condition one-hot channels.
- Target mode is region PCs, with 42 PCs per region from the 95% variance rule.
- No Gaussian temporal basis channels.
- Hidden units per region are tested at 40 and 50.
- Learning rates are tested at 1e-3, 3e-4, and 1e-4.
- Each model is trained for 25,000 iterations.
- L1 penalties are skipped. L2 penalties start at 0 and can be raised to very small values later.

The notebook compares PC reconstruction quality and firing-rate trajectories reconstructed by back-projecting the retained PCs. The firing-rate comparison is intentionally against the PC-reconstructed firing rates, not the original raw firing rates, because discarded PCs impose an irreducible reconstruction error.

## 1. Setup

In [ ]:
from pathlib import Path
from dataclasses import replace
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = next(parent for parent in Path.cwd().parents if (parent / "src").exists())

import sys
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from dal_monte_2022_analysis.ephys.modeling import (
    backproject_replay_outputs_to_firing_rates,
    load_fixation_mrnn_config,
    make_targets,
    pc_reconstructed_firing_rate_accuracy,
    reconstruction_accuracy,
    replay_fixation_mrnn_run,
    settings_from_config,
    train_fixation_mrnn_scratch,
    variance_comparison,
)


def display_figure(fig, *, dpi=95):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))


CONDITION_COLORS = {
    "face_interactive": "#b64198",
    "face_non_interactive": "#4c9a2a",
    "object": "#6f4e37",
}


## 2. Base Settings and Loss Grid

The loss configurations are intentionally incremental:

- `pc_derivatives_only`: pointwise PC reconstruction plus first- and second-difference PC losses.
- `pc_corr_var`: adds temporal correlation and variance matching in PC space.
- `pc_plus_fr`: adds Euclidean loss after PC backprojection to normalized firing-rate space.
- `pc_plus_fr_dynamics`: adds first- and second-difference losses on the PC-backprojected firing rates.

L2 scales are set to 0 here. After identifying a useful loss mix, try very small values such as `1e-6` or `1e-5`.

In [ ]:
cfg = load_fixation_mrnn_config(repo_root / "configs" / "ephys_fixation_mrnn.yaml")
base_settings = settings_from_config(cfg)
base_settings.dataset_cfg_path = str(repo_root / "configs" / "dataset.yaml")
base_settings.device = "auto"
base_settings.target_mode = "region_pcs"
base_settings.temporal_basis_count = 0
base_settings.epochs = 25_000
base_settings.activation = "softplus"
base_settings.spectral_radius = 1.0
base_settings.l1_weight_scale = 0.0
base_settings.l1_rate_scale = 0.0
base_settings.l2_weight_scale = 0.0
base_settings.l2_rate_scale = 0.0
base_settings.gradient_clip_norm = 1.0
base_settings.initialization_mode = "single"
base_settings.overwrite_seed_plan = False
base_settings.train_initial_state = True

hidden_unit_grid = [40, 50]
learning_rate_grid = [1e-3, 3e-4, 1e-4]
base_seed = 456789
scratch_prefix = "loss_sweep_region_pcs_condition_only"

loss_configs = [
    {
        "name": "pc_derivatives_only",
        "temporal_derivative_loss_scale": 1.0,
        "temporal_curvature_loss_scale": 1.0,
        "correlation_loss_scale": 0.0,
        "variance_loss_scale": 0.0,
        "fr_reconstruction_loss_scale": 0.0,
        "fr_temporal_derivative_loss_scale": 0.0,
        "fr_temporal_curvature_loss_scale": 0.0,
    },
    {
        "name": "pc_corr_var",
        "temporal_derivative_loss_scale": 1.0,
        "temporal_curvature_loss_scale": 1.0,
        "correlation_loss_scale": 0.1,
        "variance_loss_scale": 0.1,
        "fr_reconstruction_loss_scale": 0.0,
        "fr_temporal_derivative_loss_scale": 0.0,
        "fr_temporal_curvature_loss_scale": 0.0,
    },
    {
        "name": "pc_plus_fr",
        "temporal_derivative_loss_scale": 1.0,
        "temporal_curvature_loss_scale": 1.0,
        "correlation_loss_scale": 0.1,
        "variance_loss_scale": 0.1,
        "fr_reconstruction_loss_scale": 1.0,
        "fr_temporal_derivative_loss_scale": 0.0,
        "fr_temporal_curvature_loss_scale": 0.0,
    },
    {
        "name": "pc_plus_fr_dynamics",
        "temporal_derivative_loss_scale": 1.0,
        "temporal_curvature_loss_scale": 1.0,
        "correlation_loss_scale": 0.1,
        "variance_loss_scale": 0.1,
        "fr_reconstruction_loss_scale": 1.0,
        "fr_temporal_derivative_loss_scale": 1.0,
        "fr_temporal_curvature_loss_scale": 1.0,
    },
]

base_settings


## 3. Target Audit

In [ ]:
targets = make_targets(base_settings)
pc_dims = {region: targets.pcs_by_region[region].shape[-1] for region in targets.region_order}
rows = []
for region in targets.region_order:
    pca = targets.pca_by_region[region]
    rows.append({
        "region": region,
        "pc_shape": targets.pcs_by_region[region].shape,
        "pc_backprojected_fr_shape": targets.pc_reconstructed_raw_by_region()[region].shape,
        "shared_pc_dims": pc_dims[region],
        "pcs_required_for_95pct": pca.n_components_required,
        "explained_variance_in_saved_pcs": float(np.sum(pca.explained_variance_ratio)),
    })

print("Input tensor shape:", targets.input_tensor.shape)
print("Input channels: 3 condition one-hot +", targets.input_tensor.shape[-1] - 3, "temporal channels")
display(pd.DataFrame(rows))

assert targets.input_tensor.shape[-1] == 3
assert len(set(pc_dims.values())) == 1
assert next(iter(pc_dims.values())) == 42


## 4. Training and Metric Helpers

In [ ]:
def plot_loss(history, title):
    fig, ax = plt.subplots(figsize=(7.4, 3.6), dpi=130)
    for column, label in [
        ("loss", "total"),
        ("reconstruction_loss", "PC pointwise"),
        ("temporal_derivative_loss", "PC first derivative"),
        ("temporal_curvature_loss", "PC second derivative"),
        ("correlation_loss", "PC correlation"),
        ("variance_loss", "PC variance"),
        ("fr_reconstruction_loss", "FR pointwise"),
        ("fr_temporal_derivative_loss", "FR first derivative"),
        ("fr_temporal_curvature_loss", "FR second derivative"),
        ("l2_weight_loss", "L2 weight"),
        ("l2_rate_loss", "L2 rate"),
    ]:
        if column in history:
            final_value = float(history[column].iloc[-1])
            ax.plot(history["iteration"], history[column], label=f"{label}: {final_value:.4g}")
    ax.set_title(title)
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=7, ncol=2)
    display_figure(fig)
    return fig, ax


def train_loss_sweep_model(*, hidden_units, lr, loss_config):
    settings = replace(
        base_settings,
        hidden_units=int(hidden_units),
        lr=float(lr),
        seed=base_seed + int(hidden_units) + int(round(float(lr) * 1_000_000)),
        temporal_derivative_loss_scale=float(loss_config["temporal_derivative_loss_scale"]),
        temporal_curvature_loss_scale=float(loss_config["temporal_curvature_loss_scale"]),
        correlation_loss_scale=float(loss_config["correlation_loss_scale"]),
        variance_loss_scale=float(loss_config["variance_loss_scale"]),
        fr_reconstruction_loss_scale=float(loss_config["fr_reconstruction_loss_scale"]),
        fr_temporal_derivative_loss_scale=float(loss_config["fr_temporal_derivative_loss_scale"]),
        fr_temporal_curvature_loss_scale=float(loss_config["fr_temporal_curvature_loss_scale"]),
    )
    scratch_id = f"{scratch_prefix}_{loss_config['name']}_h{hidden_units:03d}_lr{lr:g}".replace(".", "p")
    result = train_fixation_mrnn_scratch(settings, scratch_id=scratch_id, overwrite=True)
    replay = replay_fixation_mrnn_run(result["run_dir"], device="cpu")
    print(
        f"loss={loss_config['name']}; hidden_units={hidden_units}; lr={lr:g}; "
        f"input_shape={replay['checkpoint']['input_tensor'].shape}; run_dir={result['run_dir']}"
    )
    return {
        "loss_config": loss_config["name"],
        "hidden_units": int(hidden_units),
        "lr": float(lr),
        "settings": settings,
        "result": result,
        "replay": replay,
    }


def _sse_and_n_by_region(replay, *, transformed=None):
    rows = []
    for region in replay["region_order"]:
        if transformed is None:
            observed = np.asarray(replay["checkpoint"]["target_by_region"][region], dtype=float)
            predicted = replay["output_by_region"][region].detach().cpu().numpy().astype(float, copy=False)
        else:
            observed, predicted = transformed(region)
        for cond_idx, condition in enumerate(replay["condition_order"]):
            residual = observed[cond_idx] - predicted[cond_idx]
            rows.append({
                "region": region,
                "condition": condition,
                "sse": float(np.sum(residual ** 2)),
                "n": int(residual.size),
            })
    return pd.DataFrame(rows)


def fit_summary_row(fit):
    replay = fit["replay"]
    history = fit["result"]["history"]
    pc_acc = reconstruction_accuracy(replay)
    fr_acc = pc_reconstructed_firing_rate_accuracy(replay)
    var = variance_comparison(replay)
    row = {
        "loss_config": fit["loss_config"],
        "hidden_units_per_region": fit["hidden_units"],
        "lr": fit["lr"],
        "final_total_loss": float(history["loss"].iloc[-1]),
        "final_pc_reconstruction_loss": float(history["reconstruction_loss"].iloc[-1]),
        "final_pc_first_derivative_loss": float(history["temporal_derivative_loss"].iloc[-1]),
        "final_pc_second_derivative_loss": float(history["temporal_curvature_loss"].iloc[-1]),
        "final_pc_correlation_loss": float(history["correlation_loss"].iloc[-1]),
        "final_pc_variance_loss": float(history["variance_loss"].iloc[-1]),
        "final_fr_reconstruction_loss": float(history["fr_reconstruction_loss"].iloc[-1]),
        "final_fr_first_derivative_loss": float(history["fr_temporal_derivative_loss"].iloc[-1]),
        "final_fr_second_derivative_loss": float(history["fr_temporal_curvature_loss"].iloc[-1]),
        "pc_mean_r2": float(pc_acc["r2"].mean()),
        "pc_mean_correlation": float(pc_acc["correlation"].mean()),
        "fr_mean_r2": float(fr_acc["r2"].mean()),
        "fr_mean_correlation": float(fr_acc["correlation"].mean()),
        "mean_pc_variance_ratio": float(var["reconstructed_to_observed_ratio"].mean()),
    }
    return row


## 5. Train Sweep

This trains 4 loss configurations x 2 hidden-unit counts x 3 learning rates = 24 models. For a quick test, reduce `loss_configs`, `hidden_unit_grid`, or `learning_rate_grid` before running.

In [ ]:
fits = []
for loss_config in loss_configs:
    for hidden_units in hidden_unit_grid:
        for lr in learning_rate_grid:
            fit = train_loss_sweep_model(hidden_units=hidden_units, lr=lr, loss_config=loss_config)
            fits.append(fit)
            plot_loss(
                fit["result"]["history"],
                f"{loss_config['name']} | h={hidden_units} | lr={lr:g}",
            )


## 6. Compare Sweep Results

In [ ]:
summary = pd.DataFrame([fit_summary_row(fit) for fit in fits])
summary = summary.sort_values(["loss_config", "hidden_units_per_region", "lr"]).reset_index(drop=True)
display(summary)

for metric in [
    "final_total_loss",
    "final_pc_reconstruction_loss",
    "final_pc_first_derivative_loss",
    "final_pc_second_derivative_loss",
    "pc_mean_correlation",
    "fr_mean_correlation",
    "mean_pc_variance_ratio",
]:
    fig, ax = plt.subplots(figsize=(7.5, 3.6), dpi=130)
    for loss_config in summary["loss_config"].unique():
        subset = summary.loc[summary["loss_config"] == loss_config]
        for hidden_units in sorted(subset["hidden_units_per_region"].unique()):
            h_subset = subset.loc[subset["hidden_units_per_region"] == hidden_units].sort_values("lr")
            ax.plot(h_subset["lr"], h_subset[metric], marker="o", label=f"{loss_config}, h={hidden_units}")
    ax.set_xscale("log")
    ax.set_xlabel("Learning rate")
    ax.set_title(metric)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=7, ncol=2)
    display_figure(fig)


## 7. PC Reconstruction Plots

Observed PC trajectories are solid. mRNN reconstructions are dotted in the same fixation-condition color.

In [ ]:
def plot_pc_reconstructions(fit, *, n_pcs=3):
    replay = fit["replay"]
    conditions = tuple(replay["condition_order"])
    regions = tuple(replay["region_order"])
    time = np.asarray(replay["checkpoint"]["timeline_s"], dtype=float)
    fig, axes = plt.subplots(len(regions), int(n_pcs), figsize=(4.0 * int(n_pcs), 2.1 * len(regions)), dpi=130, sharex=True, squeeze=False)
    for row, region in enumerate(regions):
        observed = np.asarray(replay["checkpoint"]["target_by_region"][region], dtype=float)
        predicted = replay["output_by_region"][region].detach().cpu().numpy()
        for pc_idx in range(int(n_pcs)):
            ax = axes[row, pc_idx]
            for cond_idx, condition in enumerate(conditions):
                color = CONDITION_COLORS.get(condition)
                ax.plot(time, observed[cond_idx, :, pc_idx], color=color, linewidth=1.5, linestyle="-", label=f"{condition} observed")
                ax.plot(time, predicted[cond_idx, :, pc_idx], color=color, linewidth=1.8, linestyle=":", label=f"{condition} mRNN")
            if row == 0:
                ax.set_title(f"PC{pc_idx + 1}")
            if pc_idx == 0:
                ax.set_ylabel(f"{region}\\nscore")
            if row == len(regions) - 1:
                ax.set_xlabel("Time (s)")
            ax.grid(alpha=0.2)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, fontsize=7)
    fig.suptitle(f"PC reconstructions: {fit['loss_config']} | h={fit['hidden_units']} | lr={fit['lr']:g}", y=1.02)
    fig.tight_layout()
    display_figure(fig)
    return fig, axes


# Choose fits to inspect manually after looking at the summary table.
# for fit in fits[:2]:
#     plot_pc_reconstructions(fit, n_pcs=3)


## 8. PC-Backprojected Firing-Rate Example Units

These plots compare firing-rate trajectories represented by the retained PCs. The target is not the raw observed mean firing rate; it is the observed PC scores back-projected into normalized firing-rate space. This avoids penalizing the model for information discarded by the 42-PC truncation.

In [ ]:
def plot_pc_backprojected_fr_examples(fit, *, example_unit_indices=(0, 1, 2)):
    replay = fit["replay"]
    predicted_fr = backproject_replay_outputs_to_firing_rates(replay)
    target_fr = replay["checkpoint"]["pc_reconstructed_raw_by_region"]
    conditions = tuple(replay["condition_order"])
    regions = tuple(replay["region_order"])
    time = np.asarray(replay["checkpoint"]["timeline_s"], dtype=float)
    for region in regions:
        source_features = list(replay["checkpoint"]["pca_by_region"][region]["source_features"])
        available_indices = [idx for idx in example_unit_indices if idx < predicted_fr[region].shape[-1]]
        fig, axes = plt.subplots(len(available_indices), len(conditions), figsize=(3.8 * len(conditions), 2.0 * len(available_indices)), dpi=130, sharex=True, squeeze=False)
        for row, unit_idx in enumerate(available_indices):
            for col, condition in enumerate(conditions):
                cond_idx = conditions.index(condition)
                ax = axes[row, col]
                color = CONDITION_COLORS.get(condition)
                ax.plot(time, target_fr[region][cond_idx, :, unit_idx], color=color, linewidth=1.5, linestyle="-", label="PC-backprojected target")
                ax.plot(time, predicted_fr[region][cond_idx, :, unit_idx], color=color, linewidth=1.8, linestyle=":", label="mRNN backprojection")
                if row == 0:
                    ax.set_title(condition)
                if col == 0:
                    label = source_features[unit_idx] if unit_idx < len(source_features) else f"unit {unit_idx}"
                    ax.set_ylabel(f"{region}\\n{label}")
                if row == len(available_indices) - 1:
                    ax.set_xlabel("Time (s)")
                ax.grid(alpha=0.2)
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=7)
        fig.suptitle(f"PC-backprojected FR examples: {fit['loss_config']} | h={fit['hidden_units']} | lr={fit['lr']:g}", y=1.02)
        fig.tight_layout()
        display_figure(fig)


# Choose fits to inspect manually after looking at the summary table.
# for fit in fits[:2]:
#     plot_pc_backprojected_fr_examples(fit, example_unit_indices=(0, 1, 2))


## 9. Notes for Next Pass

If reconstructions remain too slow/smooth, prioritize losses that directly compare temporal shape:

- Increase PC first-derivative and second-derivative scales before increasing pointwise loss.
- Add the PC-backprojected firing-rate derivative losses if PC-level derivatives are not enough.
- Use correlation loss to reward phase/shape match even when amplitude is slightly off.
- Use variance loss to discourage collapsed low-variance trajectories.
- Keep L1 off while debugging temporal fidelity. Try very small L2 (`1e-6` to `1e-5`) only after a loss mix reconstructs dynamics well.
- Keep gradient clipping on when testing higher learning rates or larger hidden-state dimensions.
